#  **LangChain** 멀티모달리티(Multimodality)

- **멀티모달리티** : 텍스트, 오디오, 이미지, 비디오 등 다양한 형태의 데이터를 처리하는 기술

    - **채팅 모델**: 다양한 입출력 형식 처리
    - **임베딩 모델**: 다양한 데이터 타입의 벡터 표현
    - **벡터 저장소**: 멀티모달 데이터의 임베딩 검색


## **채팅 모델의 멀티모달 기능**

- **멀티모달 입력**은 이미지, 오디오, 비디오, 파일 등 다양한 형태로 처리가 가능하며 메시지 객체를 통해 구현됨

- LangChain의 **도구(Tools)** 기능을 활용하면 URL 참조를 통해 간접적으로 멀티모달 데이터 처리가 가능

---

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

`(3) langfuase handler 설정`

In [ ]:
from langfuse.langchain import CallbackHandler

# 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

## 1. **Chat Model 입력** 

- 일부 모델에서 **이미지, 오디오, 비디오, 파일** 등의 멀티모달 입력 지원

- **모델 호환성**에 주의해야 하며, 각 모델이 지원하는 멀티모달 입력 형식을 사전에 확인해야 함 
    - 이미지 입력 지원: `gpt-4o`, `gpt-4o-mini`
    - 오디오 입력 지원: `gpt-4o-audio-preview`

- 데이터 전달 시 **base64 인코딩**이 가장 안정적이며, URL 직접 전달은 제한적으로만 지원됨

- 이미지 **크기와 형식**이 성능에 영향을 미치므로 적절한 전처리 과정이 필요함

- 참고: https://python.langchain.com/docs/integrations/chat/

`(1) URL 직접 전달 방식`



In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 이미지 처리용 모델 초기화
image_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def process_image(image_url: str, prompt: str):
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url}}
        ]
    )
    return image_model.invoke([message])

# 예시 이미지 URL
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

result = process_image(image_url, "이 이미지를 설명해주세요")
print(result.content)

`(2) 바이트 문자열 방식`



In [ ]:
import base64
import requests

# 이미지를 base64로 인코딩
def get_base64_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded_string

# 이미지를 base64 문자열로 변환
def get_image_base64_from_url(image_url: str) -> str:
    # User-Agent 헤더 추가
    headers = { 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36' }
    response = requests.get(image_url, headers=headers)
    response.raise_for_status()
    return base64.b64encode(response.content).decode('utf-8')

# 메시지 생성
def process_image(image_url: str, prompt: str):

    # 이미지를 base64 문자열로 변환
    if not image_url.startswith("http"):
        image_data = get_base64_image(image_url)
    else:
        image_data = get_image_base64_from_url(image_url)
    message= HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
            },
        ]
    )

    return image_model.invoke([message])

# 로컬 파일 처리 예시
image_path = "portrait.jpg"
result = process_image(image_path, "이 이미지를 설명해주세요")

print(result.content)

`(3) 여러 이미지 처리`



In [ ]:
import requests
from PIL import Image 
from io import BytesIO

# 이미지 URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/600px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"

headers = { 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36' }
response = requests.get(image_url, headers=headers)
img = Image.open(BytesIO(response.content))

# 이미지 출력
display(img)

In [ ]:
# 메시지 생성 
def process_images(image_paths: list, prompt: str):

    content = [{"type": "text", "text": prompt}]

    for image_path in image_paths:
        # 이미지를 base64 문자열로 변환
        if not image_url.startswith("http"):
            image_data = get_base64_image(image_url)
        else:
            image_data = get_image_base64_from_url(image_url)

        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}})

    message = HumanMessage(content=content)

    return image_model.invoke([message])

# 로컬 파일 처리 예시
image_paths = ["portrait.jpg", "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/600px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"]

result = process_images(image_paths, "이 이미지를 비교해서 설명해주세요")

print(result.content)

## 2. **도구 호출(Tool Calling)** 

- **멀티모달 모델**은 일반적인 방식으로 도구를 바인딩하여 Tool 기능을 사용할 수 있음

- Tool 호출 시 이미지 데이터와 같은 **다양한 형식의 콘텐츠 블록**을 사용 가능함

- 기존 LangChain의 Tool 시스템과 **동일한 방식**으로 멀티모달 기능을 확장하여 사용할 수 있음

In [ ]:
from typing import Literal
from langchain_core.tools import tool

@tool
def weather_tool(weather: Literal["sunny", "cloudy", "rainy"]) -> str:
    """날씨 상태를 설명하는 도구"""
    if weather == "sunny":
        return "날씨가 맑습니다."
    
    elif weather == "cloudy":
        return "날씨가 흐립니다."
    
    elif weather == "rainy":
        return "비가 옵니다."

def setup_model_with_tools(model):
    return model.bind_tools([weather_tool])

In [ ]:
import requests
from PIL import Image 
from io import BytesIO

# 이미지 URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
headers = { 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36' }
response = requests.get(image_url, headers=headers)
img = Image.open(BytesIO(response.content))

# 이미지 출력
display(img)

In [ ]:
image_data = get_image_base64_from_url(image_url)

message = HumanMessage(
    content=[
        {"type": "text", "text": "이미지의 날씨를 알려주세요"},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}
    ],
)

# 도구 사용
model_with_tools = setup_model_with_tools(image_model)
response = model_with_tools.invoke([message])
print(response.tool_calls)

In [ ]:
# 도구 실행 
weather_tool.invoke(response.tool_calls[0])

---
### **[실습] Google Gemini 멀티모달 활용**

- 다음 API 문서를 참고해서 이미지 입력을 처리합니다. 
- 참조: https://reference.langchain.com/python/integrations/langchain_google_genai/

- pip install langchain_google_genai==0.0.11

`(1) URL 직접 전달 방식`

- 이미지:  https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/600px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg



In [ ]:
import requests
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# 모델 초기화
image_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 메시지 생성 
def process_images(image_paths: list, prompt: str):

    content = [{"type": "text", "text": prompt}]
    for image_path in image_paths:
        content.append({"type": "image_url", "image_url": {"url": image_path}})

    message = HumanMessage(content=content)

    return image_model.invoke([message])


# 예시 이미지 URL
image_url =  "http://images.cocodataset.org/val2017/000000039769.jpg"
result = process_images([image_url], "이 이미지를 설명해주세요")
print(result.content)

`(2) 바이트 문자열 방식`

- 이미지: "portrait.jpg"



In [ ]:
import base64

# 이미지를 base64로 인코딩
def get_base64_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded_string
    
# 메시지 생성
def process_image(image_url: str, prompt: str):
    image_data = get_base64_image(image_url)
    message= HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
            },
        ]
    )

    return image_model.invoke([message])

# 로컬 파일 처리 예시
image_path = "portrait.jpg"
result = process_image(image_path, "이 이미지를 설명해주세요")

print(result.content)

`(3) 여러 이미지 처리`



In [ ]:
# 메시지 생성 
def process_images(image_paths: list, prompt: str):

    content = [{"type": "text", "text": prompt}]

    for image_path in image_paths:
        # URL의 경우 Base64 인코딩 처리
        if not image_path.startswith("http"):
            image_data = get_base64_image(image_path)
        else:
            image_data = get_image_base64_from_url(image_path)

        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}})

    message = HumanMessage(content=content)

    return image_model.invoke([message])

# 로컬 파일 처리 예시
image_paths = ["portrait.jpg", "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/600px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"]

result = process_images(image_paths, "이 이미지를 비교해서 설명해주세요")

print(result.content)

---
### **[실습] Ollama 멀티모달 활용**

- 다음 Ollama 모델을 다운로드하고, 랭체인 ChaiOllama 사용하여 멀티모달 처리합니다.
- 올라마 모델: https://ollama.com/library/gemma3
- 랭체인 문서: https://docs.langchain.com/oss/python/integrations/chat/ollama#multi-modal

- pip install "langchain-ollama<1.0"

In [ ]:
# 여기에 코드를 작성하세요. 

---
### **[실습] Groq 멀티모달 활용**

- 다음 Groq 모델을 다운로드하고, 랭체인 ChatGroq 사용하여 멀티모달 처리합니다.
- Groq 문서: https://console.groq.com/docs/vision

- pip install "langchain-groq<1.0"

In [ ]:
# 여기에 코드를 작성하세요. 